# 02 · Seq2seq + attention from scratch

A tiny encoder-decoder with multi-head cross-attention trained to turn linearised specs into descriptions — the architecture family behind the production T5 (`app/generator.py`), with hand-written beam search.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import torch
from spec2desc.linearise import linearise_spec
from spec2desc.attention import EncoderBlock, DecoderBlock, causal_mask
from spec2desc.beam_search import beam_search

CONFIG = dict(d_model=64, n_heads=4, d_ff=128, epochs=8, batch=16, lr=3e-3, seed=42)
torch.manual_seed(CONFIG['seed'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# corpus
rows = [json.loads(l) for l in open('../../ml/data/generated/train.jsonl')][:800]
vocab = {'<pad>': 0, '<bos>': 1, '<eos>': 2}
for r in rows:
    for tok in (linearise_spec(r['spec']) + ' ' + r['description']).lower().split():
        vocab.setdefault(tok, len(vocab))
inv = {v: k for k, v in vocab.items()}
def enc(s): return [vocab[t] for t in s.lower().split() if t in vocab]
print('vocab:', len(vocab))

In [ ]:
class TinySeq2Seq(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = torch.nn.Embedding(len(vocab), CONFIG['d_model'])
        self.enc = EncoderBlock(CONFIG['d_model'], CONFIG['n_heads'], CONFIG['d_ff'])
        self.dec = DecoderBlock(CONFIG['d_model'], CONFIG['n_heads'], CONFIG['d_ff'])
        self.out = torch.nn.Linear(CONFIG['d_model'], len(vocab))

    def forward(self, src, tgt):
        mem = self.enc(self.emb(src))
        x, attn = self.dec(self.emb(tgt), mem,
                           causal_mask(tgt.shape[1], tgt.device), return_weights=True)
        return self.out(x), attn

model = TinySeq2Seq().to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
# teacher-forced training on (spec -> description)
opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
losses = []
for epoch in range(CONFIG['epochs']):
    for i in range(0, len(rows) - CONFIG['batch'], CONFIG['batch']):
        batch = rows[i:i + CONFIG['batch']]
        src = torch.tensor([enc(linearise_spec(r['spec']))[:48] for r in batch]).to(device)
        tgt = torch.tensor([[vocab['<bos>']] + enc(r['description'])[:40] + [vocab['<eos>']] for r in batch]).to(device)
        logits, _ = model(src, tgt[:, :-1])
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss))
print('final loss:', losses[-1])

## Attention maps: which spec fields does the decoder look at?

In [ ]:
import matplotlib.pyplot as plt
src = torch.tensor([enc(linearise_spec(rows[0]['spec']))[:48]]).to(device)
tgt = torch.tensor([[vocab['<bos>']] + enc(rows[0]['description'])[:12]]).to(device)
with torch.no_grad():
    _, attn = model(src, tgt)
plt.figure(figsize=(8, 4))
plt.imshow(attn[0, 0].cpu(), aspect='auto')
plt.xlabel('spec tokens'); plt.ylabel('output steps')
plt.title('cross-attention (head 0)'); plt.show()
print('source:', ' '.join(inv[t] for t in src[0].tolist()))

## Hand-written beam search decoding

In [ ]:
def step_fn(prefix):
    tgt = torch.tensor([list(prefix)]).to(device)
    with torch.no_grad():
        mem = model.enc(model.emb(src))
        x, _ = model.dec(model.emb(tgt), mem, causal_mask(tgt.shape[1], device))
        logp = torch.log_softmax(model.out(x)[0, -1], dim=-1)
    return {i: float(logp[i]) for i in range(len(vocab))}

beams = beam_search(step_fn, bos=vocab['<bos>'], eos=vocab['<eos>'],
                    beam_width=4, max_len=24, num_return=3)
for b in beams:
    print(round(b.score, 3), ' '.join(inv[t] for t in b.tokens[1:-1]))